# Hybrid CNN + LSTM training — Google Colab (T4)

Trains the hybrid CNN+LSTM at the CLAUDE.md target config (embedding 256, hidden 512, 2 LSTM layers, 3-layer CNN over the 18-channel board, board-feature 256, dropout 0.2) on the same 300 k filter-passing games used for the LSTM run. Expected wall-clock on a free T4: ≈60–90 min.

**Run this notebook *after* `train_full_lstm_colab.ipynb`.** It re-uses the vocab built by the LSTM run from Drive so the n-gram, LSTM, and hybrid all evaluate on identical splits with identical token IDs.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. The repo must already be pushed to GitHub. Edit `GITHUB_URL` in cell 5.
3. Have a Weights & Biases API key handy if you want experiment tracking.

**What this notebook produces and persists to Drive:**
- `hybrid_full.pt` — best-by-val-loss hybrid checkpoint
- `eval_test_hybrid.txt` — top-1/3/5 + perplexity + phase breakdown on the test split

## 1. GPU + environment sanity

In [ ]:
!nvidia-smi

## 2. Mount Drive (vocab + checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DRIVE_DIR = '/content/drive/MyDrive/chess-move-prediction'
!mkdir -p "$PROJECT_DRIVE_DIR"
# Sanity-check the LSTM run's outputs are present.
!ls -lh "$PROJECT_DRIVE_DIR/vocab.json" "$PROJECT_DRIVE_DIR/lstm_full.pt" 2>&1 | head

## 3. Get the project code (must include the hybrid additions)

In [ ]:
GITHUB_URL = 'https://github.com/kornel9/chess-move-prediction'

!cd /content && rm -rf chess-move-prediction && git clone $GITHUB_URL chess-move-prediction
%cd /content/chess-move-prediction
!git rev-parse --short HEAD

## 4. Install dependencies

In [ ]:
!pip install -q chess==1.11.2 zstandard==0.25.0 wandb==0.26.1
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 5. Re-stage the data slice

Same monthly + same `N_GAMES` as the LSTM run, so `split_games(seed=42)` produces an identical train/val/test partition. We pull the vocab from Drive instead of rebuilding.

In [ ]:
MONTHLY = '2017-01'
N_GAMES = 300000

import os
os.makedirs('data/raw', exist_ok=True)
DUMP = f'data/raw/lichess_{MONTHLY}.pgn.zst'
URL = f'https://database.lichess.org/standard/lichess_db_standard_rated_{MONTHLY}.pgn.zst'

if not os.path.exists(DUMP):
    !curl -L -o $DUMP $URL
!ls -lh $DUMP

In [ ]:
!python -m src.data.make_smoke_slice \
    --src $DUMP \
    --dst data/raw/full.pgn \
    --n $N_GAMES
!ls -lh data/raw/full.pgn

In [ ]:
# Pull the LSTM-trained vocab from Drive so the hybrid uses the *same* token IDs.
!cp "$PROJECT_DRIVE_DIR/vocab.json" data/vocab.json
!ls -lh data/vocab.json

## 6. (Optional) Weights & Biases login

In [ ]:
import wandb
wandb.login()

## 7. Train the hybrid (full config)

Per-ply training, batch 256, 4 epochs (≈25 M positions per epoch — fewer epochs are sufficient than the LSTM's per-game training). Best checkpoint saved directly to Drive so a runtime disconnect can't lose progress.

In [ ]:
!python -m src.training.train_hybrid \
    --pgn data/raw/full.pgn \
    --vocab /content/drive/MyDrive/chess-move-prediction/vocab.json \
    --out /content/drive/MyDrive/chess-move-prediction/hybrid_full.pt \
    --epochs 4 \
    --batch-size 256 \
    --embedding-dim 256 \
    --hidden-dim 512 \
    --num-layers 2 \
    --dropout 0.2 \
    --board-feature-dim 256 \
    --num-workers 2 \
    --device cuda \
    --wandb

## 8. Evaluate on the test split

In [ ]:
!python -m src.training.evaluate \
    --model-type hybrid \
    --model /content/drive/MyDrive/chess-move-prediction/hybrid_full.pt \
    --pgn data/raw/full.pgn \
    --vocab /content/drive/MyDrive/chess-move-prediction/vocab.json \
    --split test \
    --device cuda \
    | tee /content/drive/MyDrive/chess-move-prediction/eval_test_hybrid.txt

## 9. What to bring back

When the run finishes, paste the contents of `eval_test_hybrid.txt` back into the local Claude Code session along with the W&B run URL. The local session has `eval_test_ngram.txt` and `eval_test_lstm.txt` already; the hybrid completes the 3-way comparison.

Also download `hybrid_full.pt` from Drive into your local repo's `checkpoints/full/` directory if you want to use the hybrid in the Streamlit demo (rename to `hybrid.pt` to match the demo path convention).